### Step 1 — Check overlay files

Confirm that the bitstream (`.bit`) and hardware metadata (`.hwh`) files exist in the selected folder and have the same base name. Their sizes are displayed for manual inspection; no timestamp or exact file size is required.

In [1]:
from pathlib import Path

folder = Path("/home/xilinx/jupyter_notebooks/snn_sdsp")
bit_file = folder / "snn_sdsp.bit"
hwh_file = folder / "snn_sdsp.hwh"

for file in (bit_file, hwh_file):
    print(f"{file.name}: exists={file.is_file()}")
    if file.is_file():
        print(f"  size={file.stat().st_size} bytes")

assert bit_file.is_file()
assert hwh_file.is_file()
assert bit_file.stem == hwh_file.stem

print("Overlay files are ready")

snn_sdsp.bit: exists=True
  size=4045700 bytes
snn_sdsp.hwh: exists=True
  size=271489 bytes
Overlay files are ready


### Step 2 — Parse hardware metadata

Parse the `.hwh` file without programming the FPGA. Check that the DMA and SNN IP blocks are present at the expected AXI addresses.

In [2]:
from pynq import Overlay

overlay = Overlay(str(bit_file), download=False)

for name, info in overlay.ip_dict.items():
    address = int(info.get("phys_addr", 0))
    print(f"{name}: address=0x{address:08X}")

assert "axi_dma_mm2s" in overlay.ip_dict
assert "snn_sdsp_0" in overlay.ip_dict
assert overlay.ip_dict["axi_dma_mm2s"]["phys_addr"] == 0x40400000
assert overlay.ip_dict["snn_sdsp_0"]["phys_addr"] == 0x43C00000

print("Hardware metadata check passed")

axi_dma_mm2s: address=0x40400000
snn_sdsp_0: address=0x43C00000
processing_system7_0: address=0x00000000
Hardware metadata check passed


### Step 3 — Program the FPGA

Download the bitstream to the programmable logic and confirm that the overlay is loaded. This step reinitializes the FPGA design, including its weight memory.

In [3]:
print("Programming FPGA...", flush=True)

overlay.download()

print("Overlay loaded:", overlay.is_loaded())
assert overlay.is_loaded()

print("FPGA programming passed")

Programming FPGA...
Overlay loaded: True
FPGA programming passed


### Step 4 — Check SNN registers

Read the SNN control and configuration registers without starting a frame. Confirm that the accelerator is idle, uses 100 time steps, and has learning, teacher input, and calcium leak disabled by default.

In [4]:
from pynq import MMIO

snn = MMIO(0x43C00000, 0x10000)

control = snn.read(0x00)
status = snn.read(0x04)
n_cycles = snn.read(0x08)
label = snn.read(0x0C)
ca_interval = snn.read(0x10)
bist_interval = snn.read(0x14)
teacher_current = snn.read(0x18)
accepted_steps = snn.read(0x50)

print(f"CONTROL: 0x{control:08X}")
print(f"STATUS: 0x{status:08X}")
print(f"N_CYCLES: {n_cycles}")
print(f"LABEL: {label}")
print(f"CA_INTERVAL: {ca_interval}")
print(f"BIST_INTERVAL: {bist_interval}")
print(f"TEACHER_CURRENT: {teacher_current}")
print(f"ACCEPTED_STEPS: {accepted_steps}")

assert status & 0x01
assert n_cycles == 100
assert (control & 0x44) == 0
assert ca_interval == 0

print("SNN register check passed")

CONTROL: 0x00000000
STATUS: 0x00000001
N_CYCLES: 100
LABEL: 0
CA_INTERVAL: 0
BIST_INTERVAL: 0
TEACHER_CURRENT: 128
ACCEPTED_STEPS: 0
SNN register check passed


### Step 5 — Check the DMA send channel

Confirm that the memory-to-stream DMA channel exists, is running, and reports no errors. The idle flag may be false before the first transfer.

In [6]:
dma = overlay.axi_dma_mm2s
dma_status = dma.mmio.read(0x04)

print("Send channel:", dma.sendchannel)
print("Receive channel:", dma.recvchannel)
print(f"DMA status: 0x{dma_status:08X}")
print("DMA running:", dma.sendchannel.running)
print("DMA idle:", dma.sendchannel.idle)

assert dma.sendchannel is not None
assert dma.recvchannel is None
assert dma.sendchannel.running
assert (dma_status & 0x70) == 0

print("DMA channel check passed")

Send channel: <pynq.lib.dma._SDMAChannel object at 0xb3864a30>
Receive channel: None
DMA status: 0x00000000
DMA running: True
DMA idle: False
DMA channel check passed


### Step 6 — Zero-input DMA test

Send one 100-step frame containing no input spikes. Verify that the DMA completes, the SNN accepts all 100 steps, no stream errors occur, and all output spike counts remain zero. Learning and teacher input are disabled.

In [7]:
import time
import numpy as np
from pynq import allocate

tx_zero = allocate(shape=(2500,), dtype=np.uint32)
tx_zero[:] = 0
tx_zero.flush()

snn.write(0x00, 0x02)
time.sleep(0.01)
snn.write(0x00, 0x01)

dma.sendchannel.transfer(tx_zero)

deadline = time.monotonic() + 10.0
while not dma.sendchannel.idle:
    dma_status = dma.mmio.read(0x04)
    if dma_status & 0x70:
        raise RuntimeError(f"DMA error: 0x{dma_status:08X}")
    if time.monotonic() > deadline:
        raise TimeoutError("DMA transfer timeout")
    time.sleep(0.001)

dma.sendchannel.wait()

deadline = time.monotonic() + 10.0
while not (snn.read(0x04) & 0x04):
    if time.monotonic() > deadline:
        raise TimeoutError("SNN completion timeout")
    time.sleep(0.001)

dma_status = dma.mmio.read(0x04)
snn_status = snn.read(0x04)
accepted = snn.read(0x50)
spike_counts = [snn.read(0x24 + 4 * i) for i in range(10)]

print(f"DMA status: 0x{dma_status:08X}")
print(f"SNN status: 0x{snn_status:08X}")
print("Accepted steps:", accepted)
print("Output spike counts:", spike_counts)

assert accepted == 100
assert (snn_status & 0x18) == 0
assert spike_counts == [0] * 10

tx_zero.close()
print("Zero-input DMA test passed")

DMA status: 0x00001002
SNN status: 0x00000005
Accepted steps: 100
Output spike counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Zero-input DMA test passed


### Step 7 — Dense-input readout test

Send a frame in which all 784 input neurons spike at every time step. Verify that the SNN accepts 100 steps without stream errors and produces nonzero output spike counts. Learning and teacher input remain disabled.

In [8]:
tx_dense = allocate(shape=(2500,), dtype=np.uint32)
tx_dense[:] = 0xFFFFFFFF
tx_dense[24::25] = 0x0000FFFF
tx_dense.flush()

snn.write(0x00, 0x02)
time.sleep(0.01)
snn.write(0x00, 0x01)

dma.sendchannel.transfer(tx_dense)

deadline = time.monotonic() + 10.0
while not dma.sendchannel.idle:
    dma_status = dma.mmio.read(0x04)
    if dma_status & 0x70:
        raise RuntimeError(f"DMA error: 0x{dma_status:08X}")
    if time.monotonic() > deadline:
        raise TimeoutError("DMA transfer timeout")
    time.sleep(0.001)

dma.sendchannel.wait()

deadline = time.monotonic() + 10.0
while not (snn.read(0x04) & 0x04):
    if time.monotonic() > deadline:
        raise TimeoutError("SNN completion timeout")
    time.sleep(0.001)

dma_status = dma.mmio.read(0x04)
snn_status = snn.read(0x04)
accepted = snn.read(0x50)
spike_counts = [snn.read(0x24 + 4 * i) for i in range(10)]
result_class = snn.read(0x20)

print(f"DMA status: 0x{dma_status:08X}")
print(f"SNN status: 0x{snn_status:08X}")
print("Accepted steps:", accepted)
print("Output spike counts:", spike_counts)
print("Result class register:", result_class)

assert accepted == 100
assert (snn_status & 0x18) == 0
assert all(count > 0 for count in spike_counts)

tx_dense.close()
print("Dense-input readout test passed")

DMA status: 0x00001002
SNN status: 0x00000005
Accepted steps: 100
Output spike counts: [99, 99, 99, 99, 99, 99, 99, 99, 99, 99]
Result class register: 0
Dense-input readout test passed
